# Homework 2 Problems and Exercises

## 0. Setup

**Before starting, ensure you've completed the setup from the README:**
- Installed dependencies with `uv sync`
- Set your OpenAI API key (required for embedding_distance function)

If you haven't done this yet, please refer to the README for detailed instructions.

In [4]:
from penngrader.grader import *
import polars as pl
import re
import os
from dotenv import load_dotenv

load_dotenv()

#PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO
#TO ASSIGN POINTS TO YOU IN OUR BACKEND
STUDENT_ID = 47349725 # YOUR PENN-ID GOES HERE AS AN INTEGER

SECRET = STUDENT_ID

HW_ID='cis2450_spr26_HW2'

In [5]:
%%writefile notebook-config.yaml

grader_api_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/TestSandbox"
grader_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"
token_generator_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/GetTokens"
token_generator_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"

Overwriting notebook-config.yaml


In [6]:
if not isinstance(STUDENT_ID, int):
    raise ValueError("STUDENT_ID must be an integer representing your Penn ID.")
if len(str(STUDENT_ID)) != 8:
    raise ValueError("STUDENT_ID must be an 8-digit integer representing your Penn ID.")
if STUDENT_ID != SECRET:
    raise ValueError("SECRET does not match STUDENT_ID. Please ensure they are the same.")

grader = PennGrader('notebook-config.yaml', HW_ID, STUDENT_ID, SECRET, "cis2450_spr26")

PennGrader initialized with Student ID: 47349725

Make sure this correct or we will not be able to store your grade


## Part 1: Approximate String Matching Functions [12 Points]

For each of the following functions, implement them in `transformations/transformations_stub.py`, then **copy** (do not import) the function code into the notebook. **The autograder will fail if you import from the stub**—your implementation must appear in the notebook cells.

### 1.1 Normalize Artist Names [3 Points]

Implement `normalize_artist_name` in `transformations/transformations_stub.py` and then copy it here:

Artist names often appear in different formats across datasets (e.g., "Taylor Swift", "TAYLOR SWIFT", "Taylor Swift feat. Ed Sheeran"). To improve matching, we need to normalize them to a canonical form.

**Your function should:**
- Convert the entire string to lowercase
- Remove featuring indicators and everything after them: "feat.", "featuring", "ft.", "with" (case-insensitive)
- Remove all punctuation except spaces
- Normalize whitespace (collapse multiple spaces to single space, strip leading/trailing)

**Hints:**
- Use `str.lower()` for case normalization
- Consider `re.sub()` for pattern-based removal
- `string.punctuation` contains common punctuation characters
- Remember to handle edge cases like multiple spaces

In [7]:
# TODO: Copy normalize_artist_name from transformations/transformations_stub.py.
# Your copied code should start with
# def normalize_artist_name(name: str) -> str:
def normalize_artist_name(name: str) -> str:
    import string

    # convert to lower
    name_result = name.lower()

    # remove featuring indicators and everything after them
    name_result = name_result.split("feat.")[0].split("ft.")[0].split("featuring")[0].split("with")[0]

    # remove all punctuation
    translator = str.maketrans('', '', string.punctuation)
    name_result = name_result.translate(translator)

    # normalize whitespace
    name_result = re.sub(pattern="\\s+", repl=" ", string=name_result)
    name_result = name_result.strip()

    return name_result

# Example test cases:
normalize_artist_name("Taylor Swift feat. Ed Sheeran")  # Should return: "taylor swift"
normalize_artist_name("ARIANA GRANDE")  # Should return: "ariana grande"
normalize_artist_name("Post Malone ft. Swae Lee")  # Should return: "post malone"

# comment out the following line after implementing the function
# raise NotImplementedError("Fill in normalize_artist_name in transformations_stub.py and test here")

'post malone'

In [8]:
grader.grade('test_normalize_artist_name', normalize_artist_name)

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 1.2 Levenshtein Distance [3 Points]

Implement `levenshtein_distance` in `transformations/transformations_stub.py` and then copy it here:

The Levenshtein (edit) distance measures how many single-character edits (insertions, deletions, substitutions) are needed to transform one string into another. This is useful for catching typos and small variations in artist names.

**Your function should:**
- Implement using dynamic programming (no external libraries)
- Create a 2D matrix where `dp[i][j]` represents the edit distance between `s1[:i]` and `s2[:j]`
- Initialize: `dp[0][j] = j` (all insertions), `dp[i][0] = i` (all deletions)
- For each cell:
  - If characters match: `dp[i][j] = dp[i-1][j-1]`
  - Otherwise: `dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + 1)`
- Return `dp[len(s1)][len(s2)]`

**Hints:**
- Handle edge cases: empty strings
- The three operations are: deletion (remove from s1), insertion (add to s1), substitution (replace in s1)
- Use nested loops to fill the matrix row by row

In [9]:
# TODO: Copy levenshtein_distance from transformations/transformations_stub.py.
# Your copied code should start with
def levenshtein_distance(s1: str, s2: str) -> int:
    if (s1 == "" or s2 == ""):
        return max(len(s1), len(s2))

    dp = [[0] * (len(s2) + 1) for _ in range(len(s1) + 1)]

    for i in range(len(dp)):
        dp[i][0] = i
    for j in range(len(dp[0])):
        dp[0][j] = j

    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + 1)

    return dp[s1.__len__()-1][s2.__len__()-1]

# Example test cases:
levenshtein_distance("kitten", "sitting")  # Should return: 3
levenshtein_distance("saturday", "sunday")  # Should return: 3
levenshtein_distance("taylor swift", "tylor swift")  # Should return: 1

# comment out the following line after implementing the function
# raise NotImplementedError("Fill in levenshtein_distance in transformations_stub.py and test here")

1

In [10]:
grader.grade('test_levenshtein_distance', levenshtein_distance)

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 1.3 Q-gram Similarity [3 Points]

Implement `qgram_similarity` in `transformations/transformations_stub.py` and then copy it here:

Q-grams (or n-grams) are sequences of q consecutive characters. Comparing the overlap of q-grams between two strings provides a measure of similarity that's robust to character reordering and works well for longer strings.

**Your function should:**
- Extract all q-grams from both strings (default q=2 for bigrams)
- Use `collections.Counter` to count occurrences of each q-gram
- Calculate Jaccard similarity: `|intersection| / |union|`
- Return a float between 0.0 (no overlap) and 1.0 (identical)

**Hints:**
- To extract q-grams from string s: `[s[i:i+q] for i in range(len(s)-q+1)]`
- For Counters A and B: intersection is `A & B`, union is `A | B`
- Use `sum(counter.values())` to get total count

In [59]:
# TODO: Copy qgram_similarity from transformations/transformations_stub.py.
# Your copied code should start with
def qgram_similarity(s1: str, s2: str, q: int = 2) -> float:
    import collections

    if (s1.__len__() == 0 or s2.__len__() == 0):
        if (s1.__len__() == s2.__len__()):
            return 1
        else: 
            return 0

    # extract all q-grams
    s1_grams = [s1[i:i+q] for i in range(len(s1)-q+1)]
    s2_grams = [s2[i:i+q] for i in range(len(s2)-q+1)]

    s1_c = collections.Counter(s1_grams)
    s2_c = collections.Counter(s2_grams)
    
    intersection = s1_c & s2_c
    union = s1_c | s2_c

    sim = intersection.__len__() / union.__len__()
    return sim

# Example test cases (Jaccard: intersection/union):
qgram_similarity("night", "nacht")  # Should return ~0.14 (1 shared bigram / 7 union)
qgram_similarity("context", "contact")  # Should return ~0.33 (3 shared / 9 union)
qgram_similarity("same", "same")  # Should return 1.0 (identical)

# comment out the following line after implementing the function
#raise NotImplementedError("Fill in qgram_similarity in transformations_stub.py and test here")

1.0

In [12]:
grader.grade('test_qgram_similarity', qgram_similarity)

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 1.4 Embedding Distance [3 Points]

Implement `embedding_distance` in `transformations/transformations_stub.py` and then copy it here:

Embedding-based similarity uses OpenAI's embedding models to capture semantic meaning. This can match artists even when their names are spelled very differently (e.g., "The Beatles" vs "Beatles, The").

**Your function should:**
- Use OpenAI's embeddings API to get embeddings for both strings
- Default model: "text-embedding-3-small"
- Calculate cosine similarity between the embeddings using numpy
- Return cosine distance: `1.0 - cosine_similarity`

**Hints:**
- Use `from openai import OpenAI` to create a client
- Set `OPENAI_API_KEY` environment variable before running
- Call `client.embeddings.create(input=text, model=model_name)` to get embeddings
- The response format is `response.data[0].embedding`
- Use `numpy.dot()` and `numpy.linalg.norm()` for cosine similarity calculation
- Consider caching embeddings to disk to avoid repeated API calls (optional but recommended)

**Note:** Lower distance = more similar (0.0 = identical, 2.0 = completely different)

In [13]:
# TODO: Copy embedding_distance from transformations/transformations_stub.py.
# Your copied code should start with
def embedding_distance(s1: str, s2: str, openai_api_key: str, model_name: str = "text-embedding-3-small") -> float:
    import openai
    import numpy

    openai.api_key = openai_api_key

    s1_embeddings = openai.embeddings.create(input = s1, model=model_name).data[0].embedding
    s2_embeddings = openai.embeddings.create(input = s2, model=model_name).data[0].embedding
    
    cosine_similarity = numpy.dot(s1_embeddings, s2_embeddings)
    return 1.0 - cosine_similarity


# Example test cases:
embedding_distance("The Beatles", "Beatles, The", os.getenv("OPENAI_API_KEY"))  # Should return low distance (~0.1-0.3)
embedding_distance("Taylor Swift", "Ed Sheeran", os.getenv("OPENAI_API_KEY"))  # Should return higher distance
embedding_distance("identical", "identical", os.getenv("OPENAI_API_KEY"))  # Should return ~0.0

# comment out the following line after implementing the function
# raise NotImplementedError("Fill in embedding_distance in transformations_stub.py and test here")

np.float64(6.155594354240179e-07)

In [14]:
grader.grade('test_embedding_distance', (embedding_distance, os.environ.get("OPENAI_API_KEY")))

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


## Part 2: Artist Matching Functions [15 points]

### 2.1 Find Matching Artists (Edit Distance) [3 Points]

Implement `find_matching_artists_edit_distance` in `transformations/transformations_stub.py` and then copy it here:

Match iTunes artist names to Spotify artist names using Levenshtein (edit) distance.

**Your function should:**
- Accept `itunes_artists` (list of strings), `spotify_df` (Polars DataFrame with `artists` column), and `threshold` (default 3)
- Get unique Spotify artists (explode and unique if `artists` is a list column)
- For each iTunes artist, find the best Spotify match using `levenshtein_distance` (compare normalized names)
- Consider it a match if distance ≤ threshold
- Return a Polars DataFrame with columns: `itunes_artist`, `spotify_artist`, `distance`, `matched`

**Hints:**
- Handle both `pl.Utf8` and list-type `artists` columns (check `spotify_df.schema['artists']`)
- Use `normalize_artist_name` before comparing

In [ ]:
# TODO: Copy find_matching_artists_edit_distance from transformations/transformations_stub.py.
# Your copied code should start with

from typing import List

def find_matching_artists_edit_distance(
    itunes_artists: List[str],
    spotify_df: pl.DataFrame,
    threshold: int = 3
) -> pl.DataFrame:
    
    # get unique spotify artists
    unique_artists = spotify_df.explode("artists").get_column("artists").unique()

    # for each itunes artist, find the best spotify match
    # also construct the lists
    i_artist_list = []
    s_artist_list = []
    distance_list = []
    matched_list = []

    for i_artist in itunes_artists:
        max_threshold = 1e10
        best_match = ""

        for s_artist in unique_artists:
            dist = levenshtein_distance(normalize_artist_name(i_artist), normalize_artist_name(s_artist))

            if (dist <= max_threshold):
                max_threshold = dist
                best_match = s_artist

        i_artist_list.append(i_artist)
        s_artist_list.append(best_match)
        distance_list.append(max_threshold)
        if (max_threshold <= threshold):
            matched_list.append(True)
        else:
            matched_list.append(False)

    return pl.DataFrame({
        'itunes_artist': i_artist_list,
        'spotify_artist': s_artist_list,
        'distance': distance_list,
        'matched': matched_list
    })

# comment out the following line after implementing the function
# raise NotImplementedError("Fill in find_matching_artists_edit_distance in transformations_stub.py and test here")

In [43]:
grader.grade('test_find_matching_artists_edit_distance', find_matching_artists_edit_distance)

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 2.2 Find Matching Artists (Q-gram) [3 Points]

Implement `find_matching_artists_qgram` in `transformations/transformations_stub.py` and then copy it here:

Same structure as 2.1, but use `qgram_similarity` instead of edit distance. Match when similarity ≥ threshold (default 0.6).

**Your function should:**
- Return a Polars DataFrame with columns: `itunes_artist`, `spotify_artist`, `similarity`, `matched`

In [ ]:
# TODO: Copy find_matching_artists_qgram from transformations/transformations_stub.py.
def find_matching_artists_qgram(
    itunes_artists: List[str],
    spotify_df: pl.DataFrame,
    threshold: float = 0.6,
    q: int = 2
) -> pl.DataFrame:
    unique_artists = spotify_df.explode("artists").get_column("artists").unique()

    # for each itunes artist, find the best spotify match
    # also construct the lists
    i_artist_list = []
    s_artist_list = []
    similarity_list = []
    matched_list = []

    for i_artist in itunes_artists:
        min_threshold = 0
        best_match = ""

        for s_artist in unique_artists:
            dist = qgram_similarity(normalize_artist_name(i_artist), normalize_artist_name(s_artist), q)

            if (dist >= min_threshold):
                min_threshold = dist
                best_match = s_artist

        i_artist_list.append(i_artist)
        s_artist_list.append(best_match)
        similarity_list.append(min_threshold)
        if (min_threshold >= threshold):
            matched_list.append(True)
        else:
            matched_list.append(False)
    
    return pl.DataFrame({
        'itunes_artist': i_artist_list,
        'spotify_artist': s_artist_list,
        'similarity': similarity_list,
        'matched': matched_list
    })

# comment out the following line after implementing the function
# raise NotImplementedError("Fill in find_matching_artists_qgram in transformations_stub.py and test here")

In [49]:
grader.grade('test_find_matching_artists_qgram', find_matching_artists_qgram)

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 2.3 Find Matching Artists (Embedding) [3 Points]

Implement `find_matching_artists_embedding` in `transformations/transformations_stub.py` and then copy it here:

Same structure as 2.1, but use `embedding_distance` (OpenAI embeddings). Match when distance ≤ threshold (default 0.3). Requires `OPENAI_API_KEY`.

**Your function should:**
- Return a Polars DataFrame with columns: `itunes_artist`, `spotify_artist`, `distance`, `matched`
- Handle empty spotify_df gracefully (return structure with matched=False)

In [ ]:
# TODO: Copy find_matching_artists_embedding from transformations/transformations_stub.py.
def find_matching_artists_embedding(
    itunes_artists: List[str],
    spotify_df: pl.DataFrame,
    openai_api_key: str,
    threshold: float = 0.3
) -> pl.DataFrame:

    # get unique spotify artists
    unique_artists = spotify_df.explode("artists").get_column("artists").unique()

    # for each itunes artist, find the best spotify match
    # also construct the lists
    i_artist_list = []
    s_artist_list = []
    distance_list = []
    matched_list = []

    for i_artist in itunes_artists:
        max_threshold = 1e10
        best_match = ""

        for s_artist in unique_artists:
            dist = embedding_distance(normalize_artist_name(i_artist), normalize_artist_name(s_artist), openai_api_key)

            if (dist <= max_threshold):
                max_threshold = dist
                best_match = s_artist

        i_artist_list.append(i_artist)
        s_artist_list.append(best_match)
        distance_list.append(max_threshold)
        if (max_threshold <= threshold):
            matched_list.append(True)
        else:
            matched_list.append(False)

    return pl.DataFrame({
        'itunes_artist': i_artist_list,
        'spotify_artist': s_artist_list,
        'distance': distance_list,
        'matched': matched_list
    })

# comment out the following line after implementing the function
# raise NotImplementedError("Fill in find_matching_artists_embedding in transformations_stub.py and test here")

In [51]:
grader.grade('test_find_matching_artists_embedding', (find_matching_artists_embedding, os.environ.get("OPENAI_API_KEY")))

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 2.4 Compare iTunes and Spotify Artists [3 Points]

Implement `compare_itunes_spotify_artists` in `transformations/transformations_stub.py` and then copy it here:

Call all three matching functions and join results on `itunes_artist`.

**Your function should:**
- Call `find_matching_artists_edit_distance`, `find_matching_artists_qgram`, and `find_matching_artists_embedding`
- Join the three DataFrames on `itunes_artist`
- Return combined DataFrame with columns: `itunes_artist`, `edit_matched`, `edit_match`, `edit_distance`, `qgram_matched`, `qgram_match`, `qgram_similarity`, `embedding_matched`, `embedding_match`, `embedding_distance`

In [82]:
# TODO: Copy compare_itunes_spotify_artists from transformations/transformations_stub.py.
def compare_itunes_spotify_artists(
    itunes_artists: List[str],
    spotify_df: pl.DataFrame,
    openai_api_key: str
) -> pl.DataFrame:

    edit_distance = find_matching_artists_edit_distance(itunes_artists, spotify_df)
    qgram = find_matching_artists_qgram(itunes_artists, spotify_df)
    embedding = find_matching_artists_embedding(itunes_artists, spotify_df, openai_api_key)

    joined = edit_distance.join(
        qgram,
        on="itunes_artist",
        how='inner'
    ).rename({
        "spotify_artist": "edit_match",
        "distance": "edit_distance",
        "matched": "edit_matched",
        "spotify_artist_right": "qgram_match",
        "similarity": "qgram_similarity",
        "matched_right": "qgram_matched"
    })

    joined = joined.join(
        embedding,
        on="itunes_artist",
        how="inner"
    ).rename({
        "spotify_artist": "embedding_match",
        "distance": "embedding_distance",
        "matched": "embedding_matched",
    })

    return joined



In [73]:
grader.grade('test_compare_itunes_spotify_artists', (compare_itunes_spotify_artists, os.environ.get("OPENAI_API_KEY")))

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 2.5 Evaluate Matching Strategy [3 Points]

Implement `evaluate_matching_strategy` in `transformations/transformations_stub.py` and then copy it here:

Compute precision, recall, and F1 for a matching strategy against ground truth.

**Your function should:**
- Accept `ground_truth_df` (columns `itunes_artist`, `spotify_artist`), `matched_df` (output of a find_matching_artists_* function with `matched` column), and `strategy` (label string)
- Compute: **precision** = TP / (TP + FP), **recall** = TP / (TP + FN), **F1** = 2 * P * R / (P + R)
- TP = ground truth pairs that appear in matched_df with matched==True
- Return dict: `{"precision": float, "recall": float, "f1": float}`. Use 0.0 when a metric is undefined (e.g., no predictions or no ground truth).

In [ ]:
# TODO: Copy evaluate_matching_strategy from transformations/transformations_stub.py.
from typing import Dict

def evaluate_matching_strategy(
    ground_truth_df: pl.DataFrame,
    matched_df: pl.DataFrame,
    strategy: str = 'edit'
) -> Dict[str, float]:

    predicted_matches = matched_df.filter((pl.col('matched') == True))

    ground_truth_df = ground_truth_df.with_columns([
        pl.col('itunes_artist').cast(pl.String),
        pl.col('spotify_artist').cast(pl.String)
    ])

    TP = ground_truth_df.join(
        predicted_matches,
        on=["itunes_artist", "spotify_artist"],
        how="inner"
    ).height

    FP = predicted_matches.join(
        ground_truth_df,
        on=["itunes_artist", "spotify_artist"],
        how="anti"
    ).height

    FN = ground_truth_df.join(
        predicted_matches,
        on=["itunes_artist", "spotify_artist"],
        how="anti"
    ).height

    if (TP + FP) == 0:
        precision = 0.0
    else:
        precision = TP / (TP + FP)

    if (TP + FN) == 0:
        recall = 0.0
    else:
        recall = TP / (TP + FN)

    if (precision + recall) == 0:
        F1 = 0.0
    else:    
        F1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": F1}

itunes_artists_1 = ["Taylor Swift", "Drake", "Beyoncé"]

spotify_df = pl.DataFrame({
    'track_id': ['t1', 't2', 't3', 't4', 't5'],
    'track_name': ['Song1', 'Song2', 'Song3', 'Song4', 'Song5'],
    'artists': [['Taylor Swift'], ['Drake'], ['Beyoncé'], ['Adele'], ['The Weeknd']],
    'album_name': ['Album1', 'Album2', 'Album3', 'Album4', 'Album5'],
    'popularity': [80, 95, 70, 88, 91],
    'track_genre': [['pop'], ['rap'], ['rnb'], ['pop'], ['rnb']],
})

gt_df = pl.DataFrame({
    'itunes_artist': ['Taylor Swift', 'Drake', 'Bruh'],
    'spotify_artist': ['Taylor Swift', 'Drake', 'Bruh'],
})

matched_df = find_matching_artists_edit_distance(itunes_artists_1, spotify_df)

evaluate_matching_strategy(gt_df, matched_df)


{'precision': 0.6666666666666666,
 'recall': 0.6666666666666666,
 'f1': 0.6666666666666666}

In [110]:
grader.grade('test_evaluate_matching_strategy', evaluate_matching_strategy)

Correct! You earned 3/3 points. You are a star!

Your submission has been successfully recorded in the gradebook.


## Part 3: Billboard Integration [30 points]

This section implements three functions for integrating Billboard Hot 100 and iTunes chart data with Spotify. Implement in the stub, then **copy** (do not import) each function into the notebook.

### 3.1 Get Billboard Songs in Spotify [10 Points]

Implement `get_billboard_songs_in_spotify` in `transformations/transformations_stub.py` and then copy it here.

This function finds which Billboard Hot 100 songs are in the Spotify dataset.
It accepts a Polars DataFrame (not a Pydantic model) for the Billboard chart data.
- Normalize titles and artists in the Billboard DataFrame
- Prepare Spotify data: select relevant columns, explode artists if list type
- Normalize Spotify titles and artists using the same function
- Inner join on normalized title and artist
- Return matches with all Billboard metadata (rank, title, artist, last_week_rank, peak_rank, weeks_on_chart) and Spotify info (track_id, track_name, artists, album_name, popularity, track_genre)

In [158]:
# TODO: implement get_billboard_songs_in_spotify in transformations/transformations_stub.py then copy here and test.
# Your copied code should start with
def get_billboard_songs_in_spotify(billboard_df: pl.DataFrame, spotify_df: pl.DataFrame) -> pl.DataFrame:

    norm_billboard = billboard_df.select(
        pl.col("rank"),
        pl.col("title").map_elements(normalize_artist_name, return_dtype=pl.String),
        pl.col("artist").map_elements(normalize_artist_name, return_dtype=pl.String),
        pl.col("last_week_rank"), pl.col("peak_rank"), pl.col("weeks_on_chart")
    )

    new_spotify_df = spotify_df.explode("artists").select(
        pl.col("track_id"),
        pl.col("track_name").map_elements(normalize_artist_name, return_dtype=pl.String),
        pl.col("artists").map_elements(normalize_artist_name, return_dtype=pl.String),
        pl.col("album_name"), pl.col("popularity"),
        pl.col("track_genre")
    )

    combined = norm_billboard.join(
        new_spotify_df,
        left_on=[pl.col("title"), pl.col("artist")],
        right_on=[pl.col("track_name"), pl.col("artists")],
        how="inner"
    ).sort("rank")
    #.with_columns((pl.col("title")).alias("track_name")).with_columns((pl.col("artist")).alias("artists"))

    combined = combined.join(
        billboard_df.select("rank", "title", "artist"),
        on="rank",
        how="inner"
    ).drop(["title", "artist"]).with_columns(
        (pl.col("title_right")
    ).alias("track_name")).with_columns((pl.col("artist_right")).alias("artists")).rename({
        "title_right": "title",
        "artist_right": "artist"
    })

    return combined

# Example test:
billboard_df = pl.DataFrame({
    'rank': [1, 2],
    'title': ['Song One', 'Song Two'],
    'artist': ['Artist A', 'Artist B'],
    'last_week_rank': [2, 1],
    'peak_rank': [1, 1],
    'weeks_on_chart': [10, 15],
})
spotify_df = pl.DataFrame({
    'track_id': ['t1', 't2', 't3'],
    'track_name': ['Song One', 'Song Two', 'Other Song'],
    'artists': [['Artist A'], ['Artist B', 'Feat C'], ['Artist D']],
    'album_name': ['Album1', 'Album2', 'Album3'],
    'popularity': [80, 90, 70],
    'track_genre': [['pop'], ['rock'], ['edm']],
})
get_billboard_songs_in_spotify(billboard_df, spotify_df)

# comment out the following line after implementing the function
# raise NotImplementedError("Implement get_billboard_songs_in_spotify in transformations_stub.py and test here")

rank,last_week_rank,peak_rank,weeks_on_chart,track_id,album_name,popularity,track_genre,title,artist,track_name,artists
i64,i64,i64,i64,str,str,i64,list[str],str,str,str,str
1,2,1,10,"""t1""","""Album1""",80,"[""pop""]","""Song One""","""Artist A""","""Song One""","""Artist A"""
2,1,1,15,"""t2""","""Album2""",90,"[""rock""]","""Song Two""","""Artist B""","""Song Two""","""Artist B"""


In [159]:
grader.grade('test_get_billboard_songs_in_spotify', get_billboard_songs_in_spotify)

Correct! You earned 10/10 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 3.2 Compare Billboard and iTunes Overlap [10 Points]

Implement `compare_billboard_itunes_overlap` in `transformations/transformations_stub.py` and then copy it here.

This function finds songs that appear in BOTH the Billboard Hot 100 AND iTunes worldwide charts.
It accepts Polars DataFrames directly for both chart datasets.
- Rename Billboard columns to billboard_rank, billboard_title, billboard_artist
- Rename iTunes columns to itunes_rank, itunes_title, itunes_artist
- Normalize titles and artists in both datasets
- Inner join to find songs appearing in BOTH charts
- Join with Spotify to get full metadata
- Return with ranks from both charts plus Spotify information

This shows which songs are globally popular across multiple charts!

In [ ]:
# TODO: implement compare_billboard_itunes_overlap in transformations/transformations_stub.py then copy here and test.
# Your copied code should start with
def compare_billboard_itunes_overlap(billboard_df: pl.DataFrame, itunes_df: pl.DataFrame, spotify_df: pl.DataFrame) -> pl.DataFrame:

    # Rename Billboard columns to billboard_rank, billboard_title, billboard_artist
    billboard_df = billboard_df.rename({
        "rank": "billboard_rank",
        "title": "billboard_title",
        "artist": "billboard_artist"
    })

    # Rename iTunes columns to itunes_rank, itunes_title, itunes_artist
    itunes_df = itunes_df.rename({
        "rank": "itunes_rank",
        "title": "itunes_title",
        "artist": "itunes_artist"
    })

    # Inner join Billboard and iTunes on normalized title and artist
    joined = billboard_df.join(
        itunes_df,
        left_on=[pl.col("billboard_title").map_elements(normalize_artist_name), pl.col("billboard_artist").map_elements(normalize_artist_name)],
        right_on=[pl.col("itunes_title").map_elements(normalize_artist_name), pl.col("itunes_artist").map_elements(normalize_artist_name)],
        how="inner"
    )

    # Prepare and normalize Spotify data (explode artists if needed)
    spotify_df = spotify_df.explode("artists")

    # Join the overlap with Spotify to get full metadata
    joined = joined.join(
        spotify_df,
        left_on=[pl.col("billboard_title").map_elements(normalize_artist_name), pl.col("billboard_artist").map_elements(normalize_artist_name)],
        right_on=[pl.col("track_name").map_elements(normalize_artist_name), pl.col("artists").map_elements(normalize_artist_name)],
        how="inner"
    ).select([
        "billboard_rank", "itunes_rank", "billboard_title", "billboard_artist",
        "last_week_rank", "peak_rank", "weeks_on_chart", "track_id", "track_name", "artists",
        "album_name", "popularity", "track_genre"
    ]).sort("billboard_rank")

    return joined

# Example test:
billboard_df = pl.DataFrame({
    'rank': [1, 2],
    'title': ['Hit Song', 'Another Hit'],
    'artist': ['Pop Star', 'Rock Band'],
    'last_week_rank': [2, 1],
    'peak_rank': [1, 1],
    'weeks_on_chart': [10, 5],
})

itunes_df = pl.DataFrame({
    'rank': [1, 3],
    'title': ['Hit Song', 'Another Hit'],
    'artist': ['Pop Star', 'Rock Band'],
})

spotify_df = pl.DataFrame({
    'track_id': ['t1', 't2'],
    'track_name': ['Hit Song', 'Another Hit'],
    'artists': [['Pop Star'], ['Rock Band']],
    'album_name': ['Album1', 'Album2'],
    'popularity': [95, 90],
    'track_genre': [['pop'], ['rock']],
})

overlap = compare_billboard_itunes_overlap(billboard_df, itunes_df, spotify_df)


In [178]:
grader.grade('test_compare_billboard_itunes_overlap', compare_billboard_itunes_overlap)

Correct! You earned 10/10 points. You are a star!

Your submission has been successfully recorded in the gradebook.


### 3.3 Get Billboard Songs by Genre [10 Points]

Implement `get_billboard_songs_by_genre` in `transformations/transformations_stub.py` and then copy it here.

This function filters Billboard Hot 100 songs by a specific genre using Spotify metadata.
It accepts a Polars DataFrame for the Billboard chart data.
- First use `get_billboard_songs_in_spotify()` to match Billboard with Spotify
- Normalize the genre parameter
- Explode the track_genre list if needed
- Filter for songs matching the normalized genre
- Return Billboard songs of that genre with full metadata

This enables genre analysis of the Billboard Hot 100!

In [ ]:
# TODO: implement get_billboard_songs_by_genre in transformations/transformations_stub.py then copy here and test.
# Your copied code should start with
def get_billboard_songs_by_genre(
    billboard_df: pl.DataFrame,
    spotify_df: pl.DataFrame,
    genre: str
) -> pl.DataFrame:

    # First use get_billboard_songs_in_spotify() to match Billboard with Spotify
    billboard_in_spotify_df = get_billboard_songs_in_spotify(billboard_df, spotify_df)

    # Normalize the input genre parameter
    genre = normalize_artist_name(genre)

    billboard_in_spotify_df = billboard_in_spotify_df.explode("track_genre").filter(
        (pl.col('track_genre').map_elements(normalize_artist_name) == genre)
    ).sort("rank")
    
    return billboard_in_spotify_df


# Example test:
billboard_df = pl.DataFrame({
    'rank': [1, 2, 3],
    'title': ['Pop Song', 'Rock Song', 'Another Pop'],
    'artist': ['Pop Star', 'Rock Band', 'Pop Artist'],
    'last_week_rank': [2, 1, 3],
    'peak_rank': [1, 1, 2],
    'weeks_on_chart': [10, 5, 8],
})

spotify_df = pl.DataFrame({
    'track_id': ['t1', 't2', 't3'],
    'track_name': ['Pop Song', 'Rock Song', 'Another Pop'],
    'artists': [['Pop Star'], ['Rock Band'], ['Pop Artist']],
    'album_name': ['Album1', 'Album2', 'Album3'],
    'popularity': [95, 90, 88],
    'track_genre': [['pop'], ['rock'], ['pop']],
})

pop_songs = get_billboard_songs_by_genre(billboard_df, spotify_df, 'pop')
print(f"Found {pop_songs.height} pop songs")

# comment out the following line after implementing the function
# raise NotImplementedError("Implement get_billboard_songs_by_genre in transformations_stub.py and test here")

shape: (2, 12)
┌──────┬────────────┬───────────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ rank ┆ last_week_ ┆ peak_rank ┆ weeks_on_c ┆ … ┆ title      ┆ artist     ┆ track_nam ┆ artists   │
│ ---  ┆ rank       ┆ ---       ┆ hart       ┆   ┆ ---        ┆ ---        ┆ e         ┆ ---       │
│ i64  ┆ ---        ┆ i64       ┆ ---        ┆   ┆ str        ┆ str        ┆ ---       ┆ str       │
│      ┆ i64        ┆           ┆ i64        ┆   ┆            ┆            ┆ str       ┆           │
╞══════╪════════════╪═══════════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 1    ┆ 2          ┆ 1         ┆ 10         ┆ … ┆ Pop Song   ┆ Pop Star   ┆ Pop Song  ┆ Pop Star  │
│ 3    ┆ 3          ┆ 2         ┆ 8          ┆ … ┆ Another    ┆ Pop Artist ┆ Another   ┆ Pop       │
│      ┆            ┆           ┆            ┆   ┆ Pop        ┆            ┆ Pop       ┆ Artist    │
└──────┴────────────┴───────────┴────────────┴───┴────────────┴────────────┴

In [185]:
grader.grade('test_get_billboard_songs_by_genre', get_billboard_songs_by_genre)

Correct! You earned 10/10 points. You are a star!

Your submission has been successfully recorded in the gradebook.
